# Pretraitement Pour L'Entrainement AQUA-ATMOS

Ce notebook prepare un jeu d'entree propre, documente et reutilisable pour la comparaison de modeles.


## 1. Chargement

**Objectif**
Charger le jeu synthetique principal et verifier qu'il contient bien les colonnes necessaires.

**Resultat attendu**
Disposer d'un DataFrame de depart pret pour les transformations.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SYNTHETIC_PATH = PROJECT_ROOT / "data" / "synthetic" / "synthetic_year.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SYNTHETIC_PATH)
df.head()


## 2. Controle De Qualite

**Objectif**
Controler les types, doublons et valeurs manquantes avant toute transformation.

**Resultat attendu**
Identifier tout point bloquant avant l'entrainement.


In [ ]:
quality_report = {
    "shape": df.shape,
    "missing_values": df.isna().sum().to_dict(),
    "duplicates": int(df.duplicated().sum()),
}

quality_report


## 3. Normalisation Des Noms De Profils

**Objectif**
Rapprocher les anciens identifiants synthetiques des identifiants de villes standardises.

**Resultat attendu**
Obtenir une colonne `city_id` stable pour les futurs modeles et dashboards.


In [ ]:
city_aliases = {
    "agadir_coastal": "agadir",
    "dakar_coastal": "dakar",
    "abidjan_coastal": "abidjan",
    "douala_coastal": "douala",
    "mombasa_coastal": "mombasa",
    "walvis_bay_coastal": "walvis_bay",
}

df["city_id"] = df["profile_name"].replace(city_aliases)
df[["profile_name", "city_id"]].drop_duplicates().sort_values(["city_id", "profile_name"])


## 4. Variables Temporelles Cycliques

**Objectif**
Encoder les cycles jour/nuit et saisonniers sans casser la periodicite.

**Resultat attendu**
Ajouter des variables robustes pour les modeles lineaires et arborescents.


In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * df["day_index"] / 365)
df["day_cos"] = np.cos(2 * np.pi * df["day_index"] / 365)

df[["hour", "hour_sin", "hour_cos", "day_index", "day_sin", "day_cos"]].head()


## 5. Variables Derivees Metier

**Objectif**
Creer quelques indicateurs interpretablement utiles pour l'entrainement.

**Resultat attendu**
Enrichir le jeu sans dupliquer inutilement les informations deja presentes.


In [ ]:
df["thermal_lift_c"] = df["temp_air_c"] - df["temp_cond_c"]
df["collector_gain_c"] = df["temp_collector_c"] - df["temp_air_c"]
df["is_daylight"] = (df["solar_wm2"] > 0).astype(int)
df["high_humidity_flag"] = (df["hr_pct"] >= 70).astype(int)
df["battery_stress_flag"] = (df["soc_battery_pct"] <= 25).astype(int)
df["reservoir_high_flag"] = (df["reservoir_level_pct"] >= 80).astype(int)

engineered_columns = [
    "thermal_lift_c",
    "collector_gain_c",
    "is_daylight",
    "high_humidity_flag",
    "battery_stress_flag",
    "reservoir_high_flag",
]
df[engineered_columns].describe().T


## 6. Encodage Des Cibles

**Objectif**
Preparer des cibles numeriques stables pour les taches de classification.

**Resultat attendu**
Obtenir des labels exploitables pour les comparaisons de modeles.


In [ ]:
sorbent_mode_mapping = {
    "veille": 0,
    "absorption": 1,
    "regeneration": 2,
}

df["sorbent_mode_label"] = df["sorbent_mode"].map(sorbent_mode_mapping)
df["heater_on_label"] = df["heater_on"].astype(int)
df["sorbent_saturated_label"] = df["sorbent_saturated"].astype(int)

df[["sorbent_mode", "sorbent_mode_label", "heater_on_label", "sorbent_saturated_label"]].head()


## 7. Colonnes D'Entree Proposees

**Objectif**
Definir explicitement la table des features de reference pour la phase de benchmark.

**Resultat attendu**
Avoir une liste de colonnes stable et auditable.


In [ ]:
feature_columns = [
    "temp_air_c",
    "hr_pct",
    "solar_wm2",
    "pv_voltage",
    "temp_collector_c",
    "temp_cond_c",
    "delta_hr_sorbent",
    "reservoir_level_pct",
    "soc_battery_pct",
    "dew_point_c",
    "humidity_ratio_gkg",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "thermal_lift_c",
    "collector_gain_c",
    "is_daylight",
    "high_humidity_flag",
    "battery_stress_flag",
    "reservoir_high_flag",
]

target_columns = [
    "vcrc_state",
    "sorbent_mode_label",
    "heater_on_label",
    "sorbent_saturated_label",
]

pd.DataFrame(
    {
        "feature_columns": pd.Series(feature_columns),
        "target_columns": pd.Series(target_columns),
    }
)


## 8. Partition Sans Fuite

**Objectif**
Construire une separation `train / validation / test` par blocs temporels.

**Resultat attendu**
Eviter qu'un modele voie indirectement le futur pendant l'apprentissage.


In [ ]:
df = df.sort_values(["city_id", "day_index", "hour"]).reset_index(drop=True)

train_mask = df["day_index"] < 255
valid_mask = (df["day_index"] >= 255) & (df["day_index"] < 310)
test_mask = df["day_index"] >= 310

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [int(train_mask.sum()), int(valid_mask.sum()), int(test_mask.sum())],
        "day_index_min": [
            int(df.loc[train_mask, "day_index"].min()),
            int(df.loc[valid_mask, "day_index"].min()),
            int(df.loc[test_mask, "day_index"].min()),
        ],
        "day_index_max": [
            int(df.loc[train_mask, "day_index"].max()),
            int(df.loc[valid_mask, "day_index"].max()),
            int(df.loc[test_mask, "day_index"].max()),
        ],
    }
)

split_summary


## 9. Export Des Jeux Prepares

**Objectif**
Sauvegarder des tables reutilisables pour les notebooks d'entrainement.

**Resultat attendu**
Produire des fichiers CSV clairs, versionnables et lisibles sans code supplementaire.


In [ ]:
prepared_df = df[["city_id", "day_index", "hour", *feature_columns, *target_columns]].copy()

train_df = prepared_df.loc[train_mask].copy()
valid_df = prepared_df.loc[valid_mask].copy()
test_df = prepared_df.loc[test_mask].copy()

prepared_path = OUTPUT_DIR / "training_dataset_prepared.csv"
train_path = OUTPUT_DIR / "train_split.csv"
valid_path = OUTPUT_DIR / "validation_split.csv"
test_path = OUTPUT_DIR / "test_split.csv"

prepared_df.to_csv(prepared_path, index=False)
train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

pd.DataFrame(
    {
        "file": [prepared_path.name, train_path.name, valid_path.name, test_path.name],
        "rows": [len(prepared_df), len(train_df), len(valid_df), len(test_df)],
    }
)


## 10. Regles Pour Le Notebook D'Entrainement

**Objectif**
Fixer les bonnes pratiques pour la prochaine etape de benchmark de modeles.

**Resultat attendu**
Une ligne directrice simple avant d'ecrire le notebook d'entrainement.


In [ ]:
training_guidelines = [
    "Comparer plusieurs modeles sur exactement les memes splits.",
    "Faire le fit des scalers et encodeurs uniquement sur le train.",
    "Conserver validation pour le choix des hyperparametres, test pour l'evaluation finale.",
    "Reporter precision, recall, F1 et matrice de confusion par cible.",
    "Garder un suivi separe des cas extremes pour verifier la robustesse.",
]

pd.DataFrame({"guideline": training_guidelines})
